# Set up

In [ ]:
project_folder = "/content/drive/MyDrive/heart_attack_in_usa"

from google.colab import drive
drive.mount('/content/drive')

%cd "{project_folder}"
!git pull

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/heart_attack_in_usa
Already up to date.


# Cài đặt môi trường

In [ ]:
!git pull
!pip install -r requirements.txt

Already up to date.
Obtaining file:///content/drive/MyDrive/heart_attack_in_usa (from -r requirements.txt (line 17))
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of imbalanced-learn to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 4.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 67.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 83.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.7/457.7 kB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 258.3/258.3 kB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.3/77.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.6/438.6 kB 30.3 MB/s eta 0:00:00
   ━━━━━

# Thư viện

In [ ]:
import pandas as pd
import numpy as np
from Mylib import myfuncs
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import plotly.express as px
import re
import os
from plotly.subplots import make_subplots

# Biểu đồ các lần train model

In [ ]:
!git pull
!python run.py model_trainer

# dc1

## Đọc dữ liệu


In [ ]:
df = myfuncs.load_python_object("artifacts/data_ingestion/train_data.pkl")

df.head()


,Age,Gender,Cholesterol,BloodPressure,HeartRate,BMI,Smoker,Diabetes,Hypertension,FamilyHistory,...,ExerciseInducedAngina,Slope,NumberOfMajorVessels,Thalassemia,PreviousHeartAttack,StrokeHistory,Residence,EmploymentStatus,MaritalStatus,Outcome
298382,36,Male,117,101,114,37.3,0,1,1,1,...,No,Upsloping,3,Normal,1,0,Rural,Retired,Widowed,Heart Attack
279375,79,Female,121,139,106,29.3,0,1,1,0,...,No,Downsloping,0,Fixed defect,0,1,Rural,Unemployed,Divorced,No Heart Attack
278789,61,Male,180,138,61,26.6,1,1,1,0,...,Yes,Upsloping,1,Reversible defect,1,0,Rural,Retired,Widowed,No Heart Attack
261903,82,Female,230,144,80,29.5,0,0,0,1,...,Yes,Downsloping,0,Normal,0,1,Suburban,Unemployed,Divorced,No Heart Attack
294404,62,Male,196,110,115,32.0,0,1,0,1,...,No,Flat,2,Reversible defect,1,0,Urban,Retired,Married,No Heart Attack


In [ ]:
df.columns

Index(['Age', 'Gender', 'Cholesterol', 'BloodPressure', 'HeartRate', 'BMI',
       'Smoker', 'Diabetes', 'Hypertension', 'FamilyHistory',
       'PhysicalActivity', 'AlcoholConsumption', 'Diet', 'StressLevel',
       'Ethnicity', 'Income', 'EducationLevel', 'Medication', 'ChestPainType',
       'ECGResults', 'MaxHeartRate', 'ST_Depression', 'ExerciseInducedAngina',
       'Slope', 'NumberOfMajorVessels', 'Thalassemia', 'PreviousHeartAttack',
       'StrokeHistory', 'Residence', 'EmploymentStatus', 'MaritalStatus',
       'Outcome'],
      dtype='object')

In [ ]:
df['MaritalStatus'].unique()

array(['Widowed', 'Divorced', 'Married', 'Single'], dtype=object)

## Ý nghĩa các cột

| Cot                       | Y nghia                                                               |  Phan loai |
| ------------------------- | --------------------------------------------------------------------- | --------- |
| Age                |                                  |  num   |
| Gender                |                                  |  nom   |
| Cholesterol                |  nồng độ Cholesterol trong máu                            |  num   |
| BloodPressure                | huyết áp                           |  num   |
| HeartRate                | nhịp tim                        |  num   |
| BMI                | chỉ số cân nặng                    |  num   |
| Smoker                | có hút thuốc không                  |  bin   |
| Diabetes                | có bị tiểu đường không              |  bin   |
| Hypertension                | có bị tăng huyết áp  không              |  bin   |
| FamilyHistory                | có tiền sử gia đình  không              |  bin   |
| PhysicalActivity                | mức độ tập thể dục            |  numcat   |
| AlcoholConsumption                | mức độ tiêu thụ rượu            |  numcat   |
| Diet                | chất lượng bữa ăn (['Unhealthy','Moderate' 'Healthy'])         |  ord   |
| StressLevel                | mức độ căng thẳng       |  numcat   |
| **Ethnicity**                | chủng tộc    |  nom   |
| Income               | thu nhập   |  num   |
| EducationLevel               | trình độ học vấn (['High School', 'College', 'Postgraduate'])  |  ord   |
| Medication               | có chữa trị trước đây chưa  |  bin   |
| ChestPainType               | loại đau ngực |  nom   |
| ECGResults               | kết quả điện tâm đồ |  nom   |
| MaxHeartRate               | nhịp tim lớn nhất |  num   |
| ST_Depression               | mức độ bất thường trong điện tâm đồ |  num   |
| ExerciseInducedAngina               | đau thắt ngực có do tập thể dục gây ra không |  bin   |
| Slope               |  |  nom   |
| NumberOfMajorVessels               | số lượng của động mạch chính  |  numcat |
| Thalassemia               | một nhóm các bệnh di truyền về máu  |  nom |
| PreviousHeartAttack               | có bị heart attack trước đây chưa  |  bin |
| StrokeHistory               | có tiền sử bị đột quy không |  bin |
| Residence               | nơi sống |  nom |
| **EmploymentStatus**               | trạng thái việc làm hiện tại |  nom |
| **MaritalStatus**              | trạng thái hôn nhân |  nom |
| Outcome     | có bị đau ngực không  |  target |

In [ ]:
# Mức độ tương quan giữa MaxHeartRate và HeartRate
df['MaxHeartRate'].corr(df['HeartRate'])

0.004726773764971571

không có mối quan hệ tuyến tính giữa 2 cột

## Xóa các cột không cần thiết




### Xóa các cột

In [ ]:
df = df.drop(
    columns=[
        "Ethnicity",
        "EmploymentStatus",
        "MaritalStatus",
    ]
)

df.columns

Index(['Age', 'Gender', 'Cholesterol', 'BloodPressure', 'HeartRate', 'BMI',
       'Smoker', 'Diabetes', 'Hypertension', 'FamilyHistory',
       'PhysicalActivity', 'AlcoholConsumption', 'Diet', 'StressLevel',
       'Income', 'EducationLevel', 'Medication', 'ChestPainType', 'ECGResults',
       'MaxHeartRate', 'ST_Depression', 'ExerciseInducedAngina', 'Slope',
       'NumberOfMajorVessels', 'Thalassemia', 'PreviousHeartAttack',
       'StrokeHistory', 'Residence', 'Outcome'],
      dtype='object')

### Tỉ lệ missing các cột

In [ ]:
null_percent = df.isnull().mean() * 100
null_percent = null_percent.sort_values(ascending=False)
null_percent


,0
Age,0.0
EducationLevel,0.0
Residence,0.0
StrokeHistory,0.0
PreviousHeartAttack,0.0
Thalassemia,0.0
NumberOfMajorVessels,0.0
Slope,0.0
ExerciseInducedAngina,0.0
ST_Depression,0.0


### Xóa các cột có tỉ lệ missing lớn

Ti le missing của các cột đều = 0-> Khong xoa cot nao het !


In [ ]:
df.shape


(52542, 21)

## Đổi tên cột

In [ ]:
df.columns

Index(['Age', 'Gender', 'Cholesterol', 'BloodPressure', 'HeartRate', 'BMI',
       'Smoker', 'Diabetes', 'Hypertension', 'FamilyHistory',
       'PhysicalActivity', 'AlcoholConsumption', 'Diet', 'StressLevel',
       'Income', 'EducationLevel', 'Medication', 'ChestPainType', 'ECGResults',
       'MaxHeartRate', 'ST_Depression', 'ExerciseInducedAngina', 'Slope',
       'NumberOfMajorVessels', 'Thalassemia', 'PreviousHeartAttack',
       'StrokeHistory', 'Residence', 'Outcome'],
      dtype='object')

In [ ]:
rename_dict = {
    "Age": "Age_num",
    "Gender": "Gender_nom",
    "Cholesterol": "Cholesterol_num",
    "BloodPressure": "BloodPressure_num",
    "HeartRate": "HeartRate_num",
    "BMI": "BMI_num",
    "Smoker": "Smoker_bin",
    "Diabetes": "Diabetes_bin",
    "Hypertension": "Hypertension_bin",
    "FamilyHistory": "FamilyHistory_bin",
    "PhysicalActivity": "PhysicalActivity_numcat",
    "AlcoholConsumption": "AlcoholConsumption_numcat",
    "Diet": "Diet_ord",
    "StressLevel": "StressLevel_numcat",
    "Income": "Income_num",
    "EducationLevel": "EducationLevel_ord",
    "Medication": "Medication_bin",
    "ChestPainType": "ChestPainType_nom",
    "ECGResults": "ECGResults_nom",
    "MaxHeartRate": "MaxHeartRate_num",
    "ST_Depression": "ST_Depression_num",
    "ExerciseInducedAngina": "ExerciseInducedAngina_bin",
    "Slope": "Slope_nom",
    "NumberOfMajorVessels": "NumberOfMajorVessels_numcat",
    "Thalassemia": "Thalassemia_nom",
    "PreviousHeartAttack": "PreviousHeartAttack_bin",
    "StrokeHistory": "StrokeHistory_bin",
    "Residence": "Residence_nom",
    "Outcome": "Outcome_target",

}


df = df.rename(columns=rename_dict)

df.columns


Index(['Age_num', 'Gender_nom', 'Cholesterol_num', 'BloodPressure_num',
       'HeartRate_num', 'BMI_num', 'Smoker_bin', 'Diabetes_bin',
       'Hypertension_bin', 'FamilyHistory_bin', 'PhysicalActivity_numcat',
       'AlcoholConsumption_numcat', 'Diet_ord', 'StressLevel_numcat',
       'Income_num', 'EducationLevel_ord', 'Medication_bin',
       'ChestPainType_nom', 'ECGResults_nom', 'MaxHeartRate_num',
       'ST_Depression_num', 'ExerciseInducedAngina_bin', 'Slope_nom',
       'NumberOfMajorVessels_numcat', 'Thalassemia_nom',
       'PreviousHeartAttack_bin', 'StrokeHistory_bin', 'Residence_nom',
       'Outcome_target'],
      dtype='object')

## Sắp xếp các cột theo đúng thứ tự

In [ ]:
numeric_cols, numericCat_cols, cat_cols, binary_cols, nominal_cols, ordinal_cols, target_col = myfuncs.get_different_types_cols_from_df_4(df)


df = df[
    numeric_cols
    + numericCat_cols
    + binary_cols
    + nominal_cols
    + ordinal_cols
    + [target_col]
]


df.head()


,Age_num,Cholesterol_num,BloodPressure_num,HeartRate_num,BMI_num,Income_num,MaxHeartRate_num,ST_Depression_num,PhysicalActivity_numcat,AlcoholConsumption_numcat,...,StrokeHistory_bin,Gender_nom,ChestPainType_nom,ECGResults_nom,Slope_nom,Thalassemia_nom,Residence_nom,Diet_ord,EducationLevel_ord,Outcome_target
298382,36,117,101,114,37.3,156701,156,1.12,0,3,...,0,Male,Atypical,Normal,Upsloping,Normal,Rural,Healthy,College,Heart Attack
279375,79,121,139,106,29.3,193989,134,3.69,6,4,...,1,Female,Non-anginal,ST-T abnormality,Downsloping,Fixed defect,Rural,Moderate,Postgraduate,No Heart Attack
278789,61,180,138,61,26.6,38928,138,0.78,5,2,...,0,Male,Typical,LV hypertrophy,Upsloping,Reversible defect,Rural,Unhealthy,College,No Heart Attack
261903,82,230,144,80,29.5,79667,137,0.86,2,4,...,1,Female,Typical,LV hypertrophy,Downsloping,Normal,Suburban,Unhealthy,High School,No Heart Attack
294404,62,196,110,115,32.0,181071,183,0.02,3,3,...,0,Male,Non-anginal,ST-T abnormality,Flat,Reversible defect,Urban,Healthy,College,No Heart Attack


## Kiểm tra kiểu dữ liệu các cột

In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 223784 entries, 298382 to 350763
Data columns (total 29 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Age_num                      223784 non-null  int64  
 1   Cholesterol_num              223784 non-null  int64  
 2   BloodPressure_num            223784 non-null  int64  
 3   HeartRate_num                223784 non-null  int64  
 4   BMI_num                      223784 non-null  float64
 5   Income_num                   223784 non-null  int64  
 6   MaxHeartRate_num             223784 non-null  int64  
 7   ST_Depression_num            223784 non-null  float64
 8   PhysicalActivity_numcat      223784 non-null  int64  
 9   AlcoholConsumption_numcat    223784 non-null  int64  
 10  StressLevel_numcat           223784 non-null  int64  
 11  NumberOfMajorVessels_numcat  223784 non-null  int64  
 12  Smoker_bin                   223784 non-null  int64  
 13 

Các cột từ 12 trở đi bị sai dữ liệu


### Chuyển kdl = kdl mong muốn + NAN


In [ ]:
for col in df.columns.tolist()[12:]:
  print(f"{col} -> {set(map(type, df[col]))}")


Smoker_bin -> {<class 'int'>}
Diabetes_bin -> {<class 'int'>}
Hypertension_bin -> {<class 'int'>}
FamilyHistory_bin -> {<class 'int'>}
Medication_bin -> {<class 'str'>}
ExerciseInducedAngina_bin -> {<class 'str'>}
PreviousHeartAttack_bin -> {<class 'int'>}
StrokeHistory_bin -> {<class 'int'>}
Gender_nom -> {<class 'str'>}
ChestPainType_nom -> {<class 'str'>}
ECGResults_nom -> {<class 'str'>}
Slope_nom -> {<class 'str'>}
Thalassemia_nom -> {<class 'str'>}
Residence_nom -> {<class 'str'>}
Diet_ord -> {<class 'str'>}
EducationLevel_ord -> {<class 'str'>}
Outcome_target -> {<class 'str'>}


Tất cả các cột đều đúng kiểu dữ liệu


## Kiểm tra nội dung các cột `binary`


In [ ]:
for col in binary_cols:
  print(f"{col} -> {df[col].unique().tolist()}")

Smoker_bin -> [0, 1]
Diabetes_bin -> [1, 0]
Hypertension_bin -> [1, 0]
FamilyHistory_bin -> [1, 0]
Medication_bin -> ['Yes', 'No']
ExerciseInducedAngina_bin -> ['No', 'Yes']
PreviousHeartAttack_bin -> [1, 0]
StrokeHistory_bin -> [0, 1]


Không có cột nào hết


## Kiểm tra nội dung các cột `nominal`


In [ ]:
for col in nominal_cols:
    print(f"{col} -> {df[col].unique().tolist()}")


Gender_nom -> ['Male', 'Female']
ChestPainType_nom -> ['Atypical', 'Non-anginal', 'Typical', 'Asymptomatic']
ECGResults_nom -> ['Normal', 'ST-T abnormality', 'LV hypertrophy']
Slope_nom -> ['Upsloping', 'Downsloping', 'Flat']
Thalassemia_nom -> ['Normal', 'Fixed defect', 'Reversible defect']
Residence_nom -> ['Rural', 'Suburban', 'Urban']


## Kiểm tra nội dung các cột `ordinal`


In [ ]:
for col in ordinal_cols:
    print(f"{col} -> {df[col].unique().tolist()}")


Diet_ord -> ['Healthy', 'Moderate', 'Unhealthy']
EducationLevel_ord -> ['College', 'Postgraduate', 'High School']


## Kiểm tra nội dung các cột `target`


In [ ]:
print(f"{target_col} -> {df[target_col].unique().tolist()}")


Outcome_target -> ['Heart Attack', 'No Heart Attack']


## Fill missing value


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="mean"), numeric_cols),
        ("numCat", SimpleImputer(strategy="most_frequent"), numericCat_cols),
        ("cat", SimpleImputer(strategy="most_frequent"), cat_cols),
        ("target", SimpleImputer(strategy="most_frequent"), [target_col]),
    ]
)

df = preprocessor.fit_transform(df)

df = pd.DataFrame(
    df, columns=numeric_cols + numericCat_cols + cat_cols + [target_col]
)

df.head()


,Age_num,Cholesterol_num,BloodPressure_num,HeartRate_num,BMI_num,Income_num,MaxHeartRate_num,ST_Depression_num,PhysicalActivity_numcat,AlcoholConsumption_numcat,...,StrokeHistory_bin,Gender_nom,ChestPainType_nom,ECGResults_nom,Slope_nom,Thalassemia_nom,Residence_nom,Diet_ord,EducationLevel_ord,Outcome_target
0,36.0,117.0,101.0,114.0,37.3,156701.0,156.0,1.12,0,3,...,0,Male,Atypical,Normal,Upsloping,Normal,Rural,Healthy,College,Heart Attack
1,79.0,121.0,139.0,106.0,29.3,193989.0,134.0,3.69,6,4,...,1,Female,Non-anginal,ST-T abnormality,Downsloping,Fixed defect,Rural,Moderate,Postgraduate,No Heart Attack
2,61.0,180.0,138.0,61.0,26.6,38928.0,138.0,0.78,5,2,...,0,Male,Typical,LV hypertrophy,Upsloping,Reversible defect,Rural,Unhealthy,College,No Heart Attack
3,82.0,230.0,144.0,80.0,29.5,79667.0,137.0,0.86,2,4,...,1,Female,Typical,LV hypertrophy,Downsloping,Normal,Suburban,Unhealthy,High School,No Heart Attack
4,62.0,196.0,110.0,115.0,32.0,181071.0,183.0,0.02,3,3,...,0,Male,Non-anginal,ST-T abnormality,Flat,Reversible defect,Urban,Healthy,College,No Heart Attack


## Chuyển đổi các cột về đúng kiểu dữ liệu


In [ ]:
df[numeric_cols] = df[numeric_cols].astype("float32")
df[numericCat_cols] = df[numericCat_cols].astype("float32")
df[cat_cols] = df[cat_cols].astype("category")
df[target_col] = df[target_col].astype("category")

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 223784 entries, 0 to 223783
Data columns (total 29 columns):
 #   Column                       Non-Null Count   Dtype   
---  ------                       --------------   -----   
 0   Age_num                      223784 non-null  float32 
 1   Cholesterol_num              223784 non-null  float32 
 2   BloodPressure_num            223784 non-null  float32 
 3   HeartRate_num                223784 non-null  float32 
 4   BMI_num                      223784 non-null  float32 
 5   Income_num                   223784 non-null  float32 
 6   MaxHeartRate_num             223784 non-null  float32 
 7   ST_Depression_num            223784 non-null  float32 
 8   PhysicalActivity_numcat      223784 non-null  float32 
 9   AlcoholConsumption_numcat    223784 non-null  float32 
 10  StressLevel_numcat           223784 non-null  float32 
 11  NumberOfMajorVessels_numcat  223784 non-null  float32 
 12  Smoker_bin                   223784 non-null

In [ ]:
df['Smoker_bin'].cat.categories

Index([0, 1], dtype='int64')

## Loại bỏ duplicates


In [ ]:
print(f"Tỉ lệ duplicates: {len(df.index[df.duplicated()]) / len(df.index) * 100}")
df = df.drop_duplicates().reset_index(drop=True)

Tỉ lệ duplicates: 0.0


In [ ]:
df.shape

(223784, 29)

## Chuẩn bị thứ tự cho các cột `ordinal`, `binary`


In [ ]:
binary_cols + ordinal_cols

['Smoker_bin',
 'Diabetes_bin',
 'Hypertension_bin',
 'FamilyHistory_bin',
 'Medication_bin',
 'ExerciseInducedAngina_bin',
 'PreviousHeartAttack_bin',
 'StrokeHistory_bin',
 'Diet_ord',
 'EducationLevel_ord']

In [ ]:
a = df['EducationLevel_ord'].unique().tolist()
a

['College', 'Postgraduate', 'High School']

In [ ]:
FEATURE_ORDINAL_DICT_DC1 = {
    "Smoker_bin": [0, 1],
    "Diabetes_bin": [0, 1 ],
    "Hypertension_bin": [0, 1 ],
    "FamilyHistory_bin": [0, 1 ],
    "Medication_bin": ['No', 'Yes' ],
    "ExerciseInducedAngina_bin": ['No', 'Yes' ],
    "PreviousHeartAttack_bin": [0, 1 ],
    "StrokeHistory_bin": [0, 1 ],
    "Diet_ord": ['Unhealthy', 'Moderate', 'Healthy' ],
    "EducationLevel_ord": ['High School', 'College', 'Postgraduate'],
}

# r

In [ ]:
df = myfuncs.load_python_object("artifacts/data_ingestion/train_data.pkl")

df.columns

Index(['Age', 'Gender', 'Cholesterol', 'BloodPressure', 'HeartRate', 'BMI',
       'Smoker', 'Diabetes', 'Hypertension', 'FamilyHistory',
       'PhysicalActivity', 'AlcoholConsumption', 'Diet', 'StressLevel',
       'Ethnicity', 'Income', 'EducationLevel', 'Medication', 'ChestPainType',
       'ECGResults', 'MaxHeartRate', 'ST_Depression', 'ExerciseInducedAngina',
       'Slope', 'NumberOfMajorVessels', 'Thalassemia', 'PreviousHeartAttack',
       'StrokeHistory', 'Residence', 'EmploymentStatus', 'MaritalStatus',
       'Outcome'],
      dtype='object')

In [ ]:
df['Outcome'].value_counts() / len(df) * 100

,count
Outcome,
No Heart Attack,50.046027
Heart Attack,49.953973


# r

In [ ]:
df = myfuncs.load_python_object("artifacts/data_transformation_dt1/train_features.pkl")

df.shape

(223784, 34)

# r

In [ ]:
df = myfuncs.load_python_object("artifacts/data_correction_dc1/data.pkl")

## Ý nghĩa các cột

| Cot                       | Y nghia                                                               |  Phan loai |
| ------------------------- | --------------------------------------------------------------------- | --------- |
| Age                |                                  |  num   |
| Gender                |                                  |  nom   |
| Cholesterol                |  nồng độ Cholesterol trong máu                            |  num   |
| BloodPressure                | huyết áp                           |  num   |
| HeartRate                | nhịp tim                        |  num   |
| BMI                | chỉ số cân nặng                    |  num   |
| Smoker                | có hút thuốc không                  |  bin   |
| Diabetes                | có bị tiểu đường không              |  bin   |
| Hypertension                | có bị tăng huyết áp  không              |  bin   |
| FamilyHistory                | có tiền sử gia đình  không              |  bin   |
| PhysicalActivity                | mức độ tập thể dục            |  numcat   |
| AlcoholConsumption                | mức độ tiêu thụ rượu            |  numcat   |
| Diet                | chất lượng bữa ăn (['Unhealthy','Moderate' 'Healthy'])         |  ord   |
| StressLevel                | mức độ căng thẳng       |  numcat   |
| Income               | thu nhập   |  num   |
| EducationLevel               | trình độ học vấn (['High School', 'College', 'Postgraduate'])  |  ord   |
| Medication               | có chữa trị trước đây chưa  |  bin   |
| ChestPainType               | loại đau ngực |  nom   |
| ECGResults               | kết quả điện tâm đồ |  nom   |
| MaxHeartRate               | nhịp tim lớn nhất |  num   |
| ST_Depression               | mức độ bất thường trong điện tâm đồ |  num   |
| ExerciseInducedAngina               | đau thắt ngực có do tập thể dục gây ra không |  bin   |
| Slope               |  |  nom   |
| NumberOfMajorVessels               | số lượng của động mạch chính  |  numcat |
| Thalassemia               | một nhóm các bệnh di truyền về máu  |  nom |
| PreviousHeartAttack               | có bị heart attack trước đây chưa  |  bin |
| StrokeHistory               | có tiền sử bị đột quy không |  bin |
| Residence               | nơi sống |  nom |
| Outcome     | có bị đau ngực không  |  target |

Về ý nghĩa thì giữ nguyên các cột

In [ ]:
numeric_cols = myfuncs.get_different_types_cols_from_df_4(df)[0]

df[numeric_cols].describe().loc[['min', 'max']].T

,min,max
Age_num,30.0,84.0
Cholesterol_num,100.0,299.0
BloodPressure_num,90.0,179.0
HeartRate_num,60.0,119.0
BMI_num,18.0,40.0
Income_num,20000.0,199999.0
MaxHeartRate_num,100.0,199.0
ST_Depression_num,0.0,5.0


In [ ]:
numericcat_cols = myfuncs.get_numericcat_cols_from_df_50(df)
table = myfuncs.get_describe_stats_for_numeric_cat_cols(df[numericcat_cols]).T
table

,min,max,median
PhysicalActivity_numcat,0.0,6.0,3.0
AlcoholConsumption_numcat,0.0,4.0,2.0
StressLevel_numcat,1.0,9.0,5.0
NumberOfMajorVessels_numcat,0.0,3.0,1.0


In [ ]:
table = myfuncs.get_correlation_between_numeric_cols_36(df, numeric_cols)

table

,item0,item1,correlation
0,Age_num,Cholesterol_num,0.005930
20,HeartRate_num,MaxHeartRate_num,0.004727
15,BloodPressure_num,Income_num,0.003779
7,Cholesterol_num,BloodPressure_num,0.003567
27,MaxHeartRate_num,ST_Depression_num,0.003388
10,Cholesterol_num,Income_num,0.003189
11,Cholesterol_num,MaxHeartRate_num,0.002513
19,HeartRate_num,Income_num,0.002208
17,BloodPressure_num,ST_Depression_num,0.002165
12,Cholesterol_num,ST_Depression_num,0.001967


Không có 2 cột numeric nào tương quan mạnh với nhau

In [ ]:
df.columns

Index(['Age_num', 'Cholesterol_num', 'BloodPressure_num', 'HeartRate_num',
       'BMI_num', 'Income_num', 'MaxHeartRate_num', 'ST_Depression_num',
       'PhysicalActivity_numcat', 'AlcoholConsumption_numcat',
       'StressLevel_numcat', 'NumberOfMajorVessels_numcat', 'Smoker_bin',
       'Diabetes_bin', 'Hypertension_bin', 'FamilyHistory_bin',
       'Medication_bin', 'ExerciseInducedAngina_bin',
       'PreviousHeartAttack_bin', 'StrokeHistory_bin', 'Gender_nom',
       'ChestPainType_nom', 'ECGResults_nom', 'Slope_nom', 'Thalassemia_nom',
       'Residence_nom', 'Diet_ord', 'EducationLevel_ord', 'Outcome_target'],
      dtype='object')

In [ ]:
myfuncs.plot_hist_box_violin_plots_for_numeric_cols_37(df, numeric_cols)

Output hidden; open in https://colab.research.google.com to view.

cột Age_num đều

cột Cholesterol_num đều

cột BloodPressure_num đều

cột HeartRate_num đều

cột BMI_num đều

cột Income_num đều

cột MaxHeartRate_num đều

cột ST_Depression_num đều



In [ ]:
table = myfuncs.get_outlier_percent_of_numeric_cols_42(df, numeric_cols)
table

,0
Age_num,0.0
Cholesterol_num,0.0
BloodPressure_num,0.0
HeartRate_num,0.0
BMI_num,0.0
Income_num,0.0
MaxHeartRate_num,0.0
ST_Depression_num,0.0


Tỉ lệ outlier của các cột numeic = 0 !!!!

In [ ]:
table =  myfuncs.get_skew_on_numeric_cols_44(df, numeric_cols)
table

,0
BMI_num,0.006193
Cholesterol_num,0.003448
Age_num,0.002860
HeartRate_num,0.002824
Income_num,0.002482
MaxHeartRate_num,0.002258
ST_Depression_num,0.001352
BloodPressure_num,0.001276


Hệ số skew rất nhỏ -> dữ liệu đều

In [ ]:
target_col = myfuncs.get_target_col_from_df_26(df)
table = myfuncs.test_relation_between_numeric_features_and_cat_target_44(df, numeric_cols, target_col)
table

/usr/local/lib/python3.11/dist-packages/Mylib/myfuncs.py:1815: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  data = df.groupby(cat_col)[numeric_col].apply(list)


,0
Income_num,0.159555
BMI_num,0.365745
BloodPressure_num,0.655289
Age_num,0.799202
HeartRate_num,0.863059
Cholesterol_num,0.916296
MaxHeartRate_num,0.925209
ST_Depression_num,0.999747


Bảng ở trên là mức độ liên quan giữa các cột numeric và cột target

In [ ]:
cat_cols = myfuncs.get_cat_cols_from_df_49(df) + myfuncs.get_numericcat_cols_from_df_50(df)

table = myfuncs.do_chi_square_test_for_categorical_cols_40(df, cat_cols)
table

/usr/local/lib/python3.11/dist-packages/Mylib/myfuncs.py:1742: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  table = df.groupby([cat_col1, cat_col2]).size().unstack()
/usr/local/lib/python3.11/dist-packages/Mylib/myfuncs.py:1742: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  table = df.groupby([cat_col1, cat_col2]).size().unstack()
/usr/local/lib/python3.11/dist-packages/Mylib/myfuncs.py:1742: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silen

,0
"(Slope_nom, AlcoholConsumption_numcat)",0.026912
"(Hypertension_bin, Residence_nom)",0.036372
"(ECGResults_nom, Diet_ord)",0.039682
"(AlcoholConsumption_numcat, NumberOfMajorVessels_numcat)",0.040393
"(PhysicalActivity_numcat, StressLevel_numcat)",0.046639
...,...
"(ChestPainType_nom, Thalassemia_nom)",0.975697
"(Hypertension_bin, Medication_bin)",0.987090
"(Residence_nom, AlcoholConsumption_numcat)",0.987715
"(Medication_bin, ECGResults_nom)",0.988635


In [ ]:
table.index[table < 0.05]

Index([                  ('Slope_nom', 'AlcoholConsumption_numcat'),
                              ('Hypertension_bin', 'Residence_nom'),
                                     ('ECGResults_nom', 'Diet_ord'),
       ('AlcoholConsumption_numcat', 'NumberOfMajorVessels_numcat'),
                  ('PhysicalActivity_numcat', 'StressLevel_numcat')],
      dtype='object')

Các cặp cột sau có phụ thuộc vào nhau:
- ('Slope_nom', 'AlcoholConsumption_numcat') *
- ('Hypertension_bin', 'Residence_nom') *
- ('ECGResults_nom', 'Diet_ord') *
- ('AlcoholConsumption_numcat', 'NumberOfMajorVessels_numcat') *
- ('PhysicalActivity_numcat', 'StressLevel_numcat') *

Bỏ các cột: AlcoholConsumption_numcat, Residence_nom, Diet_ord, PhysicalActivity_numcat

In [ ]:
df = df.drop(columns = ["AlcoholConsumption_numcat", "Residence_nom", 'Diet_ord', 'PhysicalActivity_numcat'])

In [ ]:
target_col = myfuncs.get_target_col_from_df_26(df)
cat_cols = myfuncs.get_cat_cols_from_df_49(df) + myfuncs.get_numericcat_cols_from_df_50(df)

table = myfuncs.do_chi_square_test_between_categorical_cols_and_target_col_41(df, cat_cols, target_col)

table

/usr/local/lib/python3.11/dist-packages/Mylib/myfuncs.py:1742: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  table = df.groupby([cat_col1, cat_col2]).size().unstack()
/usr/local/lib/python3.11/dist-packages/Mylib/myfuncs.py:1742: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  table = df.groupby([cat_col1, cat_col2]).size().unstack()
/usr/local/lib/python3.11/dist-packages/Mylib/myfuncs.py:1742: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silen

,0
Thalassemia_nom,0.060542
PreviousHeartAttack_bin,0.076708
Gender_nom,0.087831
ECGResults_nom,0.297433
StressLevel_numcat,0.316314
Medication_bin,0.332736
EducationLevel_ord,0.363844
Diabetes_bin,0.453265
ChestPainType_nom,0.506845
NumberOfMajorVessels_numcat,0.741585


Các cột đều độc lập với cột target vì p_value > 0.05

# dc2

## Đọc dữ liệu


In [ ]:
df = myfuncs.load_python_object("artifacts/data_ingestion/train_data.pkl")

df.head()


,Age,Gender,Cholesterol,BloodPressure,HeartRate,BMI,Smoker,Diabetes,Hypertension,FamilyHistory,...,ExerciseInducedAngina,Slope,NumberOfMajorVessels,Thalassemia,PreviousHeartAttack,StrokeHistory,Residence,EmploymentStatus,MaritalStatus,Outcome
298382,36,Male,117,101,114,37.3,0,1,1,1,...,No,Upsloping,3,Normal,1,0,Rural,Retired,Widowed,Heart Attack
279375,79,Female,121,139,106,29.3,0,1,1,0,...,No,Downsloping,0,Fixed defect,0,1,Rural,Unemployed,Divorced,No Heart Attack
278789,61,Male,180,138,61,26.6,1,1,1,0,...,Yes,Upsloping,1,Reversible defect,1,0,Rural,Retired,Widowed,No Heart Attack
261903,82,Female,230,144,80,29.5,0,0,0,1,...,Yes,Downsloping,0,Normal,0,1,Suburban,Unemployed,Divorced,No Heart Attack
294404,62,Male,196,110,115,32.0,0,1,0,1,...,No,Flat,2,Reversible defect,1,0,Urban,Retired,Married,No Heart Attack


In [ ]:
df.columns

Index(['Age', 'Gender', 'Cholesterol', 'BloodPressure', 'HeartRate', 'BMI',
       'Smoker', 'Diabetes', 'Hypertension', 'FamilyHistory',
       'PhysicalActivity', 'AlcoholConsumption', 'Diet', 'StressLevel',
       'Ethnicity', 'Income', 'EducationLevel', 'Medication', 'ChestPainType',
       'ECGResults', 'MaxHeartRate', 'ST_Depression', 'ExerciseInducedAngina',
       'Slope', 'NumberOfMajorVessels', 'Thalassemia', 'PreviousHeartAttack',
       'StrokeHistory', 'Residence', 'EmploymentStatus', 'MaritalStatus',
       'Outcome'],
      dtype='object')

## Ý nghĩa các cột

| Cot                       | Y nghia                                                               |  Phan loai |
| ------------------------- | --------------------------------------------------------------------- | --------- |
| Age                |                                  |  num   |
| Gender                |                                  |  nom   |
| Cholesterol                |  nồng độ Cholesterol trong máu                            |  num   |
| BloodPressure                | huyết áp                           |  num   |
| HeartRate                | nhịp tim                        |  num   |
| BMI                | chỉ số cân nặng                    |  num   |
| Smoker                | có hút thuốc không                  |  bin   |
| Diabetes                | có bị tiểu đường không              |  bin   |
| Hypertension                | có bị tăng huyết áp  không              |  bin   |
| FamilyHistory                | có tiền sử gia đình  không              |  bin   |
| PhysicalActivity                | mức độ tập thể dục            |  numcat   |
| AlcoholConsumption                | mức độ tiêu thụ rượu            |  numcat   |
| Diet                | chất lượng bữa ăn (['Unhealthy','Moderate' 'Healthy'])         |  ord   |
| StressLevel                | mức độ căng thẳng       |  numcat   |
| **Ethnicity**                | chủng tộc    |  nom   |
| Income               | thu nhập   |  num   |
| EducationLevel               | trình độ học vấn (['High School', 'College', 'Postgraduate'])  |  ord   |
| Medication               | có chữa trị trước đây chưa  |  bin   |
| ChestPainType               | loại đau ngực |  nom   |
| ECGResults               | kết quả điện tâm đồ |  nom   |
| MaxHeartRate               | nhịp tim lớn nhất |  num   |
| ST_Depression               | mức độ bất thường trong điện tâm đồ |  num   |
| ExerciseInducedAngina               | đau thắt ngực có do tập thể dục gây ra không |  bin   |
| Slope               |  |  nom   |
| NumberOfMajorVessels               | số lượng của động mạch chính  |  numcat |
| Thalassemia               | một nhóm các bệnh di truyền về máu  |  nom |
| PreviousHeartAttack               | có bị heart attack trước đây chưa  |  bin |
| StrokeHistory               | có tiền sử bị đột quy không |  bin |
| Residence               | nơi sống |  nom |
| **EmploymentStatus**               | trạng thái việc làm hiện tại |  nom |
| **MaritalStatus**              | trạng thái hôn nhân |  nom |
| Outcome     | có bị đau ngực không  |  target |

## Xóa các cột không cần thiết




### Xóa các cột

In [ ]:
df.columns

Index(['Age', 'Gender', 'Cholesterol', 'BloodPressure', 'HeartRate', 'BMI',
       'Smoker', 'Diabetes', 'Hypertension', 'FamilyHistory',
       'PhysicalActivity', 'AlcoholConsumption', 'Diet', 'StressLevel',
       'Ethnicity', 'Income', 'EducationLevel', 'Medication', 'ChestPainType',
       'ECGResults', 'MaxHeartRate', 'ST_Depression', 'ExerciseInducedAngina',
       'Slope', 'NumberOfMajorVessels', 'Thalassemia', 'PreviousHeartAttack',
       'StrokeHistory', 'Residence', 'EmploymentStatus', 'MaritalStatus',
       'Outcome'],
      dtype='object')

In [ ]:
df = df.drop(
    columns=[
        "Ethnicity",
        "EmploymentStatus",
        "MaritalStatus",
        "AlcoholConsumption",
        "Residence",
        "Diet",
        "PhysicalActivity",
    ]
)

df.columns

Index(['Age', 'Gender', 'Cholesterol', 'BloodPressure', 'HeartRate', 'BMI',
       'Smoker', 'Diabetes', 'Hypertension', 'FamilyHistory', 'StressLevel',
       'Income', 'EducationLevel', 'Medication', 'ChestPainType', 'ECGResults',
       'MaxHeartRate', 'ST_Depression', 'ExerciseInducedAngina', 'Slope',
       'NumberOfMajorVessels', 'Thalassemia', 'PreviousHeartAttack',
       'StrokeHistory', 'Outcome'],
      dtype='object')

### Tỉ lệ missing các cột

In [ ]:
null_percent = df.isnull().mean() * 100
null_percent = null_percent.sort_values(ascending=False)
null_percent


,0
Age,0.0
Medication,0.0
StrokeHistory,0.0
PreviousHeartAttack,0.0
Thalassemia,0.0
NumberOfMajorVessels,0.0
Slope,0.0
ExerciseInducedAngina,0.0
ST_Depression,0.0
MaxHeartRate,0.0


### Xóa các cột có tỉ lệ missing lớn

Ti le missing của các cột đều = 0-> Khong xoa cot nao het !


In [ ]:
df.shape


(52542, 21)

## Đổi tên cột

In [ ]:
df.columns

Index(['Age', 'Gender', 'Cholesterol', 'BloodPressure', 'HeartRate', 'BMI',
       'Smoker', 'Diabetes', 'Hypertension', 'FamilyHistory', 'StressLevel',
       'Income', 'EducationLevel', 'Medication', 'ChestPainType', 'ECGResults',
       'MaxHeartRate', 'ST_Depression', 'ExerciseInducedAngina', 'Slope',
       'NumberOfMajorVessels', 'Thalassemia', 'PreviousHeartAttack',
       'StrokeHistory', 'Outcome'],
      dtype='object')

In [ ]:
rename_dict = {
    "Age": "Age_num",
    "Gender": "Gender_nom",
    "Cholesterol": "Cholesterol_num",
    "BloodPressure": "BloodPressure_num",
    "HeartRate": "HeartRate_num",
    "BMI": "BMI_num",
    "Smoker": "Smoker_bin",
    "Diabetes": "Diabetes_bin",
    "Hypertension": "Hypertension_bin",
    "FamilyHistory": "FamilyHistory_bin",
    "StressLevel": "StressLevel_numcat",
    "Income": "Income_num",
    "EducationLevel": "EducationLevel_ord",
    "Medication": "Medication_bin",
    "ChestPainType": "ChestPainType_nom",
    "ECGResults": "ECGResults_nom",
    "MaxHeartRate": "MaxHeartRate_num",
    "ST_Depression": "ST_Depression_num",
    "ExerciseInducedAngina": "ExerciseInducedAngina_bin",
    "Slope": "Slope_nom",
    "NumberOfMajorVessels": "NumberOfMajorVessels_numcat",
    "Thalassemia": "Thalassemia_nom",
    "PreviousHeartAttack": "PreviousHeartAttack_bin",
    "StrokeHistory": "StrokeHistory_bin",
    "Outcome": "Outcome_target",

}


df = df.rename(columns=rename_dict)

df.columns


Index(['Age_num', 'Gender_nom', 'Cholesterol_num', 'BloodPressure_num',
       'HeartRate_num', 'BMI_num', 'Smoker_bin', 'Diabetes_bin',
       'Hypertension_bin', 'FamilyHistory_bin', 'StressLevel_numcat',
       'Income_num', 'EducationLevel_ord', 'Medication_bin',
       'ChestPainType_nom', 'ECGResults_nom', 'MaxHeartRate_num',
       'ST_Depression_num', 'ExerciseInducedAngina_bin', 'Slope_nom',
       'NumberOfMajorVessels_numcat', 'Thalassemia_nom',
       'PreviousHeartAttack_bin', 'StrokeHistory_bin', 'Outcome_target'],
      dtype='object')

## Sắp xếp các cột theo đúng thứ tự

In [ ]:
numeric_cols, numericCat_cols, cat_cols, binary_cols, nominal_cols, ordinal_cols, target_col = myfuncs.get_different_types_cols_from_df_4(df)


df = df[
    numeric_cols
    + numericCat_cols
    + binary_cols
    + nominal_cols
    + ordinal_cols
    + [target_col]
]


df.head()


,Age_num,Cholesterol_num,BloodPressure_num,HeartRate_num,BMI_num,Income_num,MaxHeartRate_num,ST_Depression_num,StressLevel_numcat,NumberOfMajorVessels_numcat,...,ExerciseInducedAngina_bin,PreviousHeartAttack_bin,StrokeHistory_bin,Gender_nom,ChestPainType_nom,ECGResults_nom,Slope_nom,Thalassemia_nom,EducationLevel_ord,Outcome_target
298382,36,117,101,114,37.3,156701,156,1.12,2,3,...,No,1,0,Male,Atypical,Normal,Upsloping,Normal,College,Heart Attack
279375,79,121,139,106,29.3,193989,134,3.69,1,0,...,No,0,1,Female,Non-anginal,ST-T abnormality,Downsloping,Fixed defect,Postgraduate,No Heart Attack
278789,61,180,138,61,26.6,38928,138,0.78,5,1,...,Yes,1,0,Male,Typical,LV hypertrophy,Upsloping,Reversible defect,College,No Heart Attack
261903,82,230,144,80,29.5,79667,137,0.86,5,0,...,Yes,0,1,Female,Typical,LV hypertrophy,Downsloping,Normal,High School,No Heart Attack
294404,62,196,110,115,32.0,181071,183,0.02,8,2,...,No,1,0,Male,Non-anginal,ST-T abnormality,Flat,Reversible defect,College,No Heart Attack


## Kiểm tra kiểu dữ liệu các cột

In [ ]:
df.info()


<class 'pandas.core.frame.DataFrame'>
Index: 223784 entries, 298382 to 350763
Data columns (total 25 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Age_num                      223784 non-null  int64  
 1   Cholesterol_num              223784 non-null  int64  
 2   BloodPressure_num            223784 non-null  int64  
 3   HeartRate_num                223784 non-null  int64  
 4   BMI_num                      223784 non-null  float64
 5   Income_num                   223784 non-null  int64  
 6   MaxHeartRate_num             223784 non-null  int64  
 7   ST_Depression_num            223784 non-null  float64
 8   StressLevel_numcat           223784 non-null  int64  
 9   NumberOfMajorVessels_numcat  223784 non-null  int64  
 10  Smoker_bin                   223784 non-null  int64  
 11  Diabetes_bin                 223784 non-null  int64  
 12  Hypertension_bin             223784 non-null  int64  
 13 

Các cột từ 10 trở đi bị sai dữ liệu


### Chuyển kdl = kdl mong muốn + NAN


In [ ]:
for col in df.columns.tolist()[10:]:
  print(f"{col} -> {set(map(type, df[col]))}")


Smoker_bin -> {<class 'int'>}
Diabetes_bin -> {<class 'int'>}
Hypertension_bin -> {<class 'int'>}
FamilyHistory_bin -> {<class 'int'>}
Medication_bin -> {<class 'str'>}
ExerciseInducedAngina_bin -> {<class 'str'>}
PreviousHeartAttack_bin -> {<class 'int'>}
StrokeHistory_bin -> {<class 'int'>}
Gender_nom -> {<class 'str'>}
ChestPainType_nom -> {<class 'str'>}
ECGResults_nom -> {<class 'str'>}
Slope_nom -> {<class 'str'>}
Thalassemia_nom -> {<class 'str'>}
EducationLevel_ord -> {<class 'str'>}
Outcome_target -> {<class 'str'>}


Tất cả các cột đều đúng kiểu dữ liệu


## Kiểm tra nội dung các cột `binary`


In [ ]:
for col in binary_cols:
  print(f"{col} -> {df[col].unique().tolist()}")

Smoker_bin -> [0, 1]
Diabetes_bin -> [1, 0]
Hypertension_bin -> [1, 0]
FamilyHistory_bin -> [1, 0]
Medication_bin -> ['Yes', 'No']
ExerciseInducedAngina_bin -> ['No', 'Yes']
PreviousHeartAttack_bin -> [1, 0]
StrokeHistory_bin -> [0, 1]


Không có cột nào hết


## Kiểm tra nội dung các cột `nominal`


In [ ]:
for col in nominal_cols:
    print(f"{col} -> {df[col].unique().tolist()}")


Gender_nom -> ['Male', 'Female']
ChestPainType_nom -> ['Atypical', 'Non-anginal', 'Typical', 'Asymptomatic']
ECGResults_nom -> ['Normal', 'ST-T abnormality', 'LV hypertrophy']
Slope_nom -> ['Upsloping', 'Downsloping', 'Flat']
Thalassemia_nom -> ['Normal', 'Fixed defect', 'Reversible defect']


## Kiểm tra nội dung các cột `ordinal`


In [ ]:
for col in ordinal_cols:
    print(f"{col} -> {df[col].unique().tolist()}")


EducationLevel_ord -> ['College', 'Postgraduate', 'High School']


## Kiểm tra nội dung các cột `target`


In [ ]:
print(f"{target_col} -> {df[target_col].unique().tolist()}")


Outcome_target -> ['Heart Attack', 'No Heart Attack']


## Fill missing value


In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="mean"), numeric_cols),
        ("numCat", SimpleImputer(strategy="most_frequent"), numericCat_cols),
        ("cat", SimpleImputer(strategy="most_frequent"), cat_cols),
        ("target", SimpleImputer(strategy="most_frequent"), [target_col]),
    ]
)

df = preprocessor.fit_transform(df)

df = pd.DataFrame(
    df, columns=numeric_cols + numericCat_cols + cat_cols + [target_col]
)

df.head()


,Age_num,Cholesterol_num,BloodPressure_num,HeartRate_num,BMI_num,Income_num,MaxHeartRate_num,ST_Depression_num,StressLevel_numcat,NumberOfMajorVessels_numcat,...,ExerciseInducedAngina_bin,PreviousHeartAttack_bin,StrokeHistory_bin,Gender_nom,ChestPainType_nom,ECGResults_nom,Slope_nom,Thalassemia_nom,EducationLevel_ord,Outcome_target
0,36.0,117.0,101.0,114.0,37.3,156701.0,156.0,1.12,2,3,...,No,1,0,Male,Atypical,Normal,Upsloping,Normal,College,Heart Attack
1,79.0,121.0,139.0,106.0,29.3,193989.0,134.0,3.69,1,0,...,No,0,1,Female,Non-anginal,ST-T abnormality,Downsloping,Fixed defect,Postgraduate,No Heart Attack
2,61.0,180.0,138.0,61.0,26.6,38928.0,138.0,0.78,5,1,...,Yes,1,0,Male,Typical,LV hypertrophy,Upsloping,Reversible defect,College,No Heart Attack
3,82.0,230.0,144.0,80.0,29.5,79667.0,137.0,0.86,5,0,...,Yes,0,1,Female,Typical,LV hypertrophy,Downsloping,Normal,High School,No Heart Attack
4,62.0,196.0,110.0,115.0,32.0,181071.0,183.0,0.02,8,2,...,No,1,0,Male,Non-anginal,ST-T abnormality,Flat,Reversible defect,College,No Heart Attack


## Chuyển đổi các cột về đúng kiểu dữ liệu


In [ ]:
df[numeric_cols] = df[numeric_cols].astype("float32")
df[numericCat_cols] = df[numericCat_cols].astype("float32")
df[cat_cols] = df[cat_cols].astype("category")
df[target_col] = df[target_col].astype("category")

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 223784 entries, 0 to 223783
Data columns (total 25 columns):
 #   Column                       Non-Null Count   Dtype   
---  ------                       --------------   -----   
 0   Age_num                      223784 non-null  float32 
 1   Cholesterol_num              223784 non-null  float32 
 2   BloodPressure_num            223784 non-null  float32 
 3   HeartRate_num                223784 non-null  float32 
 4   BMI_num                      223784 non-null  float32 
 5   Income_num                   223784 non-null  float32 
 6   MaxHeartRate_num             223784 non-null  float32 
 7   ST_Depression_num            223784 non-null  float32 
 8   StressLevel_numcat           223784 non-null  float32 
 9   NumberOfMajorVessels_numcat  223784 non-null  float32 
 10  Smoker_bin                   223784 non-null  category
 11  Diabetes_bin                 223784 non-null  category
 12  Hypertension_bin             223784 non-null

## Loại bỏ duplicates


In [ ]:
print(f"Tỉ lệ duplicates: {len(df.index[df.duplicated()]) / len(df.index) * 100}")
df = df.drop_duplicates().reset_index(drop=True)

Tỉ lệ duplicates: 0.0


In [ ]:
df.shape

(223784, 25)

## Chuẩn bị thứ tự cho các cột `ordinal`, `binary`


In [ ]:
binary_cols + ordinal_cols

['Smoker_bin',
 'Diabetes_bin',
 'Hypertension_bin',
 'FamilyHistory_bin',
 'Medication_bin',
 'ExerciseInducedAngina_bin',
 'PreviousHeartAttack_bin',
 'StrokeHistory_bin',
 'EducationLevel_ord']

In [ ]:
a = df['EducationLevel_ord'].unique().tolist()
a

['College', 'Postgraduate', 'High School']

In [ ]:
FEATURE_ORDINAL_DICT_DC2 = {
    "Smoker_bin": [0, 1],
    "Diabetes_bin": [0, 1 ],
    "Hypertension_bin": [0, 1 ],
    "FamilyHistory_bin": [0, 1 ],
    "Medication_bin": ['No', 'Yes' ],
    "ExerciseInducedAngina_bin": ['No', 'Yes' ],
    "PreviousHeartAttack_bin": [0, 1 ],
    "StrokeHistory_bin": [0, 1 ],
    "EducationLevel_ord": ['High School', 'College', 'Postgraduate'],
}

# r

In [ ]:
df = myfuncs.load_python_object("artifacts/data_transformation_dt3/train_features.pkl")

df.head()

,pca0,pca1,pca2,pca3,pca4,pca5,pca6,pca7,pca8,pca9,...,pca223,pca224,pca225,pca226,pca227,pca228,pca229,pca230,pca231,pca232
0,-1.661526,2.058058,3.213584,-0.765631,-1.741805,-1.670164,1.451049,1.046390,-2.466914,0.582640,...,-0.408463,0.111695,-0.011799,-0.020577,-0.029478,-0.110192,0.038664,0.031137,0.156643,0.526698
1,-0.912667,-0.882463,-1.953631,0.070687,-0.061787,1.704281,0.866892,-0.219715,-1.255511,0.851052,...,0.082122,-0.066082,0.204786,0.062744,-0.113750,0.046424,0.056756,0.076023,0.030001,-0.049963
2,1.197142,3.162825,-0.447362,-0.835603,2.579973,-0.762039,1.060638,-0.908541,0.740528,-1.858598,...,-0.101089,0.195725,0.017062,0.072467,0.039320,0.265026,-0.062954,0.075800,-0.080829,-0.037750
3,-1.709482,-1.806622,-0.046619,-0.680429,1.313744,-0.454545,-1.055899,1.637428,0.487950,-0.055176,...,0.016330,0.203650,0.141175,0.034872,0.036613,-0.257881,0.032493,-0.151054,0.122524,0.056755
4,1.990224,-0.545641,-2.394142,1.787902,-0.684521,2.322648,1.264722,2.496718,0.779047,2.297332,...,-0.105927,0.087820,-0.014572,0.216124,0.145835,0.123639,-0.167010,0.072222,-0.111671,0.095576


In [ ]:
target = myfuncs.load_python_object("artifacts/data_transformation_dt3/train_target.pkl")
target

,Outcome_target
0,0
1,1
2,1
3,1
4,1
...,...
223779,0
223780,0
223781,1
223782,0


In [ ]:
df = myfuncs.load_python_object("artifacts/data_transformation_dt3_batch_50000/")